In [1]:
# A default setup cell.
# It imports environment variables, define 'devtools.debug" as a buildins, set PYTHONPATH, and code auto-reload
# Copy it in other Notebooks

%load_ext autoreload
%autoreload 2
%reset -f

from devtools import debug  # noqa: F401  # noqa: F811
from dotenv import load_dotenv
from rich import print  # noqa: F401

assert load_dotenv(verbose=True)

In [2]:
from src.utils.config_mngr import global_config, global_config_reload

global_config_reload()

list_demos = global_config().merge_with("config/demos/document_extractor.yaml").get_list("Document_extractor_demo")


test_schema = next((item for item in list_demos if item.get("schema_name") == "Rainbow File"))


In [3]:
from src.demos.ekg.retriever_tool_factory import PydanticRag

KV_STORE = "file"


vector_store_factory = PydanticRag.get_vector_store_factory()


rag = PydanticRag(
    model_definition=test_schema,
    vector_store_factory=vector_store_factory,
    llm_id=None,
    kv_store_id=KV_STORE,
)

vector_store_factory.delete_collection()

2025-08-19 17:24:57.712 | DEBUG    | src.ai_core.llm_factory:get_llm:588 - get LLM:'kimi_k2_openrouter' -extra: {'temperature': 0.0}
2025-08-19 17:24:57.955 | DEBUG    | src.ai_extra.pgvector_factory:create_pg_vector_store:77 - Hybrid search enabled with config: HybridSearchConfig(tsv_column='content_tsv', tsv_lang='pg_catalog.english', fts_query='', fusion_function=<function reciprocal_rank_fusion at 0x77dd20bceb60>, fusion_function_parameters={}, primary_top_k=4, secondary_top_k=4, index_name='pydantic_fields_qwen3_06b_tsv_index', index_type='GIN')
2025-08-19 17:24:57.989 | DEBUG    | src.ai_extra.pgvector_factory:create_pg_vector_store:93 - Use existing pgvector table : pydantic_fields_qwen3_06b


src/ai_extra/pgvector_factory.py:107 create_pg_vector_store
    vector_store: <langchain_postgres.v2.vectorstores.PGVectorStore object at 0x77dcfea672c0> (PGVectorStore)


2025-08-19 17:24:58.040 | DEBUG    | src.ai_extra.pgvector_factory:create_pg_vector_store:116 - Failed to apply hybrid search index: 'PGVectorStore' object has no attribute 'apply_hybrid_search_index'
2025-08-19 17:24:58.041 | DEBUG    | src.ai_core.vector_store_factory:get:228 - get vector store  : PgVector/pydantic_fields_qwen3_06b
2025-08-19 17:24:58.042 | INFO     | src.ai_core.vector_store_factory:delete_collection:319 - drop file public.pydantic_fields_qwen3_06b


In [4]:
import os
from pathlib import Path

doc_id = "03.RESM-SOL-9000559500_CNES_TMA_VENUS_VIP_PEPS_THEIA_MUSCATE-v0.2"

file1 = Path(os.getenv("ONEDRIVE", "")) / "prj/atos-kg/rainbow-json/" / (doc_id + "_extracted.json")
assert file1.exists()
doc_text = file1.read_text()

rainbow_report = rag.analyze_document(
    document_id=doc_id,
    markdown=doc_text,
)

print("Structured result:", rainbow_report)

assert rainbow_report


2025-08-19 17:24:58.123 | DEBUG    | src.ai_extra.kv_store_factory:load_object:148 - read 'RainbowProjectAnalysis/03.RESM-SOL-9000559500_CNES_TMA_VENUS_VIP_PEPS_THEIA_MUSCATE-v0.2.json' from KV store
2025-08-19 17:24:58.123 | DEBUG    | src.ai_extra.kv_store_factory:load_object:150 - stored_data: {'identification': {'name': 'CNES TMA VENUS VIP_PEPS_ THEIA MUSCATE', 'opportunity_id': '9000559500', 'customer': 'CNES', 'customer_segment': 'Aerospace', 'status': 'Solution Review', 'start_date': '2019-04-01', 'end_date': '2022-03-31'}, 'description': {'objectives': ['Tierce Maintenance Applicative (TMA) of CNES Earth-observation data production centres and their image-processing chains', 'Corrective and evolutionary maintenance of three lots: VENUS VIP, PEPS, THEIA MUSCATE'], 'scope': 'Take-over phase followed by corrective/evolutionary maintenance for three lots (mono-award framework)', 'success_metrics': ['SLA compliance', 'Quality of deliverables', 'Customer satisfaction'], 'differentiat

Structured result:
RainbowProjectAnalysis(
    identification=ProjectIdentification(
        name='CNES TMA VENUS VIP_PEPS_ THEIA MUSCATE',
        opportunity_id='9000559500',
        customer='CNES',
        customer_segment='Aerospace',
        status='Solution Review',
        start_date='2019-04-01',
        end_date='2022-03-31'
    ),
    description=ProjectDescription(
        objectives=[
            'Tierce Maintenance Applicative (TMA) of CNES Earth-observation data production centres and their 
image-processing chains',
            'Corrective and evolutionary maintenance of three lots: VENUS VIP, PEPS, THEIA MUSCATE'
        ],
        scope='Take-over phase followed by corrective/evolutionary maintenance for three lots (mono-award 
framework)',
        success_metrics=['SLA compliance', 'Quality of deliverables', 'Customer satisfaction'],
        differentiators=[
            'Existing incumbent on PEPS',
            'Deep Earth-observation domain knowledge',
            'MUNDI platform synergies'
        ]
    ),
    team=[
        PersonRole(name='Marc Ferrer', role='Global Client Leader', organization='Atos', contact_type='internal'),
        PersonRole(
            name='Aurore Dorez',
            role='Sales Lead / Deal Maker',
            organization='Atos',
            contact_type='internal'
        ),
        PersonRole(name='Barthélémy Marti', role='Bid Manager', organization='Atos', contact_type='internal'),
        PersonRole(
            name='Olivier Rondeau',
            role='Solution Manager / Contract Executive',
            organization='Atos',
            contact_type='internal'
        ),
        PersonRole(name='Caroline Jaulin', role='Finance Lead', organization='Atos', contact_type='internal'),
        PersonRole(name='Danièle Phankongsy', role='Deal Lawyer', organization='Atos', contact_type='internal'),
        PersonRole(name='Sonia Gouel', role='Project Manager', organization='Atos', contact_type='internal'),
        PersonRole(
            name='Pierre Bourrousse',
            role='Strategy Achat Sponsor',
            organization='CNES',
            contact_type='external'
        ),
        PersonRole(name='Gérard Lassalle-Valler', role='Sponsor', organization='CNES', contact_type='external'),
        PersonRole(
            name='Sylvia Sylvander',
            role='CP CNES Décideur',
            organization='CNES',
            contact_type='external'
        )
    ],
    delivery=DeliveryInfo(
        business_lines=['B1P S BS Toulouse'],
        locations=['Toulouse, France'],
        partners=[
            'ACS (subcontractor for Venus VIP maintenance, imposed by CNES)',
            'PIXSTART (subcontractor for THEIA MUSCATE first-year fixes and knowledge transfer)'
        ],
        technologies=[
            'VENUS VIP image-processing chains',
            'PEPS Sentinel data exploitation platform',
            'THEIA MUSCATE continental-surface data services'
        ]
    ),
    financials=FinancialMetrics(
        tcv=1800000.0,
        annual_revenue=300000.0,
        project_margin=21.0,
        payment_terms='30 days from invoice date, quarterly invoicing'
    ),
    risks=[
        RiskAnalysis(
            risk_description='Penalties due to SLA non-compliance caused by quality or delivery delays',
            impact_level='high',
            mitigation_strategy='Rigorous QA and project management',
            status='open'
        ),
        RiskAnalysis(
            risk_description='Cost overruns from underestimation of rework, missing packages, or non-representative
platforms',
            impact_level='medium',
            mitigation_strategy='Detailed due-diligence and contingency planning',
            status='open'
        ),
        RiskAnalysis(
            risk_description='Competency retention issues due to turnover and declining activity',
            impact_level='medium',
            mitigation_strategy='Knowledge management and retention incentives'

In [5]:
chunks = rag.chunck(rainbow_report)
debug(chunks)


/tmp/ipykernel_118375/280282971.py:2 <module>
    chunks: [
        Document(
            metadata={
                'field_name': 'identification',
                'model_class': 'RainbowProjectAnalysis',
                'description': 'Project identification information',
                'document_id': '03.RESM-SOL-9000559500_CNES_TMA_VENUS_VIP_PEPS_THEIA_MUSCATE-v0.2_extracted.json',
                'entity_id': '9000559500',
            },
            page_content=(
                '# name\n'
                'CNES TMA VENUS VIP_PEPS_ THEIA MUSCATE\n'
                '# opportunity_id\n'
                '9000559500\n'
                '# customer\n'
                'CNES\n'
                '# customer_segment\n'
                'Aerospace\n'
                '# status\n'
                'Solution Review\n'
                '# start_date\n'
                '2019-04-01\n'
                '# end_date\n'
                '2022-03-31'
            ),
        ),
        Document(
            m

[Document(metadata={'field_name': 'identification', 'model_class': 'RainbowProjectAnalysis', 'description': 'Project identification information', 'document_id': '03.RESM-SOL-9000559500_CNES_TMA_VENUS_VIP_PEPS_THEIA_MUSCATE-v0.2_extracted.json', 'entity_id': '9000559500'}, page_content='# name\nCNES TMA VENUS VIP_PEPS_ THEIA MUSCATE\n# opportunity_id\n9000559500\n# customer\nCNES\n# customer_segment\nAerospace\n# status\nSolution Review\n# start_date\n2019-04-01\n# end_date\n2022-03-31'),
 Document(metadata={'field_name': 'description', 'model_class': 'RainbowProjectAnalysis', 'description': 'Project characteristics description', 'document_id': '03.RESM-SOL-9000559500_CNES_TMA_VENUS_VIP_PEPS_THEIA_MUSCATE-v0.2_extracted.json', 'entity_id': '9000559500'}, page_content='- objectives\n  - Tierce Maintenance Applicative (TMA) of CNES Earth-observation data production centres and their image-processing chains'),
 Document(metadata={'field_name': 'description', 'model_class': 'RainbowProjectA

In [6]:
from langchain_core.utils.function_calling import convert_to_openai_tool

dyn_tool = rag.create_vector_search_tool()
debug(convert_to_openai_tool(dyn_tool))


2025-08-19 17:24:58.246 | DEBUG    | src.ai_extra.pgvector_factory:create_pg_vector_store:77 - Hybrid search enabled with config: HybridSearchConfig(tsv_column='content_tsv', tsv_lang='pg_catalog.english', fts_query='', fusion_function=<function reciprocal_rank_fusion at 0x77dd20bceb60>, fusion_function_parameters={}, primary_top_k=4, secondary_top_k=4, index_name='pydantic_fields_qwen3_06b_tsv_index', index_type='GIN')
2025-08-19 17:24:58.291 | INFO     | src.ai_extra.pgvector_factory:create_pg_vector_store:88 - pgvector vector table created: table_name='pydantic_fields_qwen3_06b' schema_name='public'
2025-08-19 17:24:58.292 | INFO     | src.ai_extra.pgvector_factory:create_pg_vector_store:90 - Hybrid search configured with TSV column: content_tsv


src/ai_extra/pgvector_factory.py:107 create_pg_vector_store
    vector_store: <langchain_postgres.v2.vectorstores.PGVectorStore object at 0x77dcfd327770> (PGVectorStore)


2025-08-19 17:24:58.316 | DEBUG    | src.ai_extra.pgvector_factory:create_pg_vector_store:116 - Failed to apply hybrid search index: 'PGVectorStore' object has no attribute 'apply_hybrid_search_index'
2025-08-19 17:24:58.316 | DEBUG    | src.ai_core.vector_store_factory:get:228 - get vector store  : PgVector/pydantic_fields_qwen3_06b


/tmp/ipykernel_118375/1577126679.py:4 <module>
    convert_to_openai_tool(dyn_tool): {
        'type': 'function',
        'function': {
            'name': 'RainbowProjectAnalysis_retriever',
            'description': (
                '\n'
                "Retrieve information related to documents described as 'Input document for a review meeting (called '"
                "Rainbow') that assess an opportunity for a project and the business proposal, before it's sent to the"
                ' customer..\n'
                "Each document is related to a unique id 'opportunity_id', with is typically a  Unique identifier from"
                ' Salesforce or similar CRM.\n'
                'Argument are:\n'
                '- expanded query.  \n'
                '- selected_sections: a list of section names that best match the query. Select several if you are not'
                ' sure.\n'
                "- entity_keys:  list of 'opportunity_id' mentionned in the context / discussion

{'type': 'function',
 'function': {'name': 'RainbowProjectAnalysis_retriever',
  'description': "\nRetrieve information related to documents described as 'Input document for a review meeting (called 'Rainbow') that assess an opportunity for a project and the business proposal, before it's sent to the customer..\nEach document is related to a unique id 'opportunity_id', with is typically a  Unique identifier from Salesforce or similar CRM.\nArgument are:\n- expanded query.  \n- selected_sections: a list of section names that best match the query. Select several if you are not sure.\n- entity_keys:  list of 'opportunity_id' mentionned in the context / discussion..\n",
  'parameters': {'properties': {'query': {'description': '\nThe query to the semantic search vector store. \nProvide several variants of the request to improve the semantic matching \n(ex: broaden, examples,...) ',
     'type': 'string'},
    'selected_sections': {'description': "\n                    List of sections relev

In [7]:
r = dyn_tool.invoke({"query": "CNES", "selected_sections": ["team"], "entity_keys": []})
print(r)

No information found

In [8]:
rag.get_top_class()

src.utils.pydantic.dyn_model_factory.RainbowProjectAnalysis

In [9]:
rag.kv_to_vector_store()



2025-08-19 17:24:59.527 | INFO     | src.ai_core.vector_store_factory:delete_collection:319 - drop file public.pydantic_fields_qwen3_06b
2025-08-19 17:24:59.563 | DEBUG    | src.ai_extra.kv_store_factory:load_object:148 - read 'RainbowProjectAnalysis/L3-MTN__Group_-_MTN_ecommerce-AFR-9000992602-OFF_Ver2.1__1_.json' from KV store
2025-08-19 17:24:59.564 | DEBUG    | src.ai_extra.kv_store_factory:load_object:150 - stored_data: {'identification': {'name': 'eCommerce', 'opportunity_id': '9000992602', 'customer': 'MTN Group', 'customer_segment': 'TMT - Telecom', 'status': 'PUR to HND', 'start_date': '2022-08-18', 'end_date': 'Q4 2022'}, 'description': {'objectives': ['Offer overview', 'Approval to submit offer to MTN as prime'], 'scope': 'Implementation of Optimizely Digital Experience Platform across 18 MTN Opcos in Africa, covering software licensing, professional services, training and managed services for 5 years.', 'success_metrics': ['Successful submission of prime offer', 'Contract s

ProgrammingError: (sqlalchemy.dialects.postgresql.asyncpg.ProgrammingError) <class 'asyncpg.exceptions.UndefinedTableError'>: relation "public.pydantic_fields_qwen3_06b" does not exist
[SQL: INSERT INTO "public"."pydantic_fields_qwen3_06b"("langchain_id", "content", "embedding", "content_tsv", "entity_id", "field_name", "model_class", "langchain_metadata")VALUES ($1, $2, $3, to_tsvector('pg_catalog.english', $4), $5, $6, $7, $8) ON CONFLICT ("langchain_id") DO UPDATE SET "content" = EXCLUDED."content", "embedding" = EXCLUDED."embedding", "content_tsv" = EXCLUDED."content_tsv", "langchain_metadata" = EXCLUDED."langchain_metadata", "entity_id" = EXCLUDED."entity_id", "field_name" = EXCLUDED."field_name", "model_class" = EXCLUDED."model_class";]
[parameters: ('3d3f68d4-26ea-407f-82b3-b559df207771', '# name\neCommerce\n# opportunity_id\n9000992602\n# customer\nMTN Group\n# customer_segment\nTMT - Telecom\n# status\nPUR to HND\n# start_date\n2022-08-18\n# end_date\nQ4 2022', '[-0.0567384026

In [ ]:
# 2. Index the document
rag.store_chunks(chunks)
print("Document stored.")


In [ ]:
hits = rag.query_vectorstore("e-mail address", k=2)
print("Vector hits:", hits)

In [ ]:
# 3. Query the vector store
hits = rag.query_vectorstore("revenue", k=2, filter={"field_name": {"$eq": "financials"}})
print("Vector hits:", hits)

In [ ]:
rag.get_key_description()

In [ ]:
# vector_store_factory.delete_collection()